In [1]:
from pathlib import Path
import sys,json,hashlib
ROOT = Path.cwd().resolve().parent
sys.path.insert(0,str(ROOT/'src'))
from sneak.fcg_registry import FCGRegistry,verify_rows
from sneak.merkle import canonical,leaf,root,verify
r=FCGRegistry(ROOT/'data/fcg/sneak-016-replay.jsonl')
print(ROOT)

/Users/byron/projects/hackathon/sneak-audit-fco-20261020


In [2]:
rows=r.rows()
assert len(rows)==101
roots={58:'f0579f511bccecf09c588f1d564e33a12af3409502a56b7868c4b61eab87587a',94:'e214642bb3e8153c8a2a16a0b63444e6534afb95cc0c5853e808948df9903e78',101:'40a1df7aac73fcdc5897752a342823cf9aa312ae7fb29bf0f0c401b64332587d'}
for n, expected in roots.items(): assert verify_rows(rows[:n])['mmr_root']==expected
print('PASS: all historical MMR prefixes',r.verify())

PASS: all historical MMR prefixes {'verified': True, 'fco_count': 101, 'mmr_size': 198, 'mmr_root': '40a1df7aac73fcdc5897752a342823cf9aa312ae7fb29bf0f0c401b64332587d', 'fcg_edges': 113}


In [3]:
p=json.loads((ROOT/'receipts/connectors/SNEAK-021_REGISTERED_LOMO.json').read_text())
assert p['checkpoint_leaf_count']==8
assert len(p['scored_breakpoints'])==8
assert all(x['scan_outcome']=='NOT_TESTED' and x['application_score'] is None for x in p['scored_breakpoints'])
assert sum(x['delta_fco_id'] is not None for x in p['scored_breakpoints'])==4
c=p['checkpoint_chain']; hh=[leaf(x['record']) for x in c['leaves']]
assert root(hh).hex()==c['root']
assert all(verify(h,x['proof'],bytes.fromhex(c['root'])) for h,x in zip(hh,c['leaves']))
print('PASS: 8 Merkle+score attempts, 4 ablation FCOs, all coverage unaccepted')

PASS: 8 Merkle+score attempts, 4 ablation FCOs, all coverage unaccepted


In [4]:
s=json.loads((ROOT/'receipts/connectors/SNEAK-022_SECURITY_REVIEW.json').read_text())
assert hashlib.sha256(canonical(s['statement'])).hexdigest()==s['statement_sha256']
assert s['statement']['sast']['count']==5 and s['statement']['delta']['delta']==-2
assert s['statement']['sca']['status']=='FAILED'
assert s['statement']['javier']['application_score'] is None
assert s['successor_mmr']==r.verify()
assert r.inclusion_proof(rows[-1]['fco_id'])['verified']
print('PASS: Snyk evidence, no clean score, last FCO inclusion')

PASS: Snyk evidence, no clean score, last FCO inclusion
